In [ ]:
import pandas as pd

In [ ]:
folderPath = "/mnt/d/Personal project/DeepLearning/nnForGraphs/attackClassifier/UNSW-NB15/"

In [ ]:
dataDF = pd.read_csv(f"{folderPath}UNSW-NB15_1_cleaned_encoded.csv")

In [ ]:
dataDF.head()

In [ ]:
dataDF.drop(columns=["Unnamed: 0"] , inplace=True)

In [ ]:
import networkx as netx

graph = netx.MultiDiGraph()

In [ ]:
src = dataDF['srcip'].to_list()
dest = dataDF['dstip'].to_list()

In [ ]:
graph.add_nodes_from(src)
graph.add_nodes_from(dest)

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "cpu"
)
print(f"Using device: {device}")


In [ ]:
from torch_geometric.utils import from_networkx

for _, row in dataDF.iterrows():
    graph.add_edge(
        row["srcip"],
        row["dstip"],
        sport=row["sport"],
        dsport=row["dsport"],
        proto=row["proto"],
        state=row["state"],
        dur=row["dur"],
        sbytes=row["sbytes"],
        dbytes=row["dbytes"],
        Label=row["Label"]
    )

data = from_networkx(graph).to(device)

In [ ]:
edge_features = []
edge_labels = []

for _, _, attr in graph.edges(data=True):

    edge_features.append([
        attr["dur"],
        attr["sbytes"],
        attr["dbytes"],
        attr["sport"],
        attr["dsport"],
        attr["proto"],
        attr["state"]
    ])

    edge_labels.append(attr["Label"])

In [ ]:
from sklearn.preprocessing import StandardScaler
scalerEdges = StandardScaler()
edge_features[:] = scalerEdges.fit_transform(edge_features)

In [ ]:
data.edge_attr = torch.tensor(
    edge_features,
    dtype=torch.float
).to(device)

data.edge_label = torch.tensor(
    edge_labels,
    dtype=torch.long
).to(device)

In [ ]:
from sklearn.model_selection import train_test_split

edge_indices = torch.arange(data.edge_index.size(1))

trainIndx, testIndx = train_test_split(
    edge_indices.cpu().numpy(),
    test_size=0.25,
    random_state=31,
    stratify=data.edge_label.cpu().numpy()
)

trainIndx = torch.tensor(trainIndx, dtype=torch.long)
testIndx = torch.tensor(testIndx, dtype=torch.long)

print("Train edges:", len(trainIndx))
print("Test edges:", len(testIndx))

In [ ]:
trainDF = dataDF.iloc[trainIndx.cpu().numpy()]

In [ ]:
import pandas as pd

src_features = trainDF.groupby("srcip").agg(
    out_degree=("srcip", "count"),
    avg_sbytes=("sbytes", "mean"),
    avg_spkts=("Spkts", "mean"),
    avg_sload=("Sload", "mean"),
    avg_sttl=("sttl", "mean"),
    avg_duration=("dur", "mean")
)

dst_features = trainDF.groupby("dstip").agg(
    in_degree=("dstip", "count"),
    avg_dbytes=("dbytes", "mean"),
    avg_dpkts=("Dpkts", "mean"),
    avg_dload=("Dload", "mean"),
    avg_dttl=("dttl", "mean")
)

In [ ]:
src_features

In [ ]:
dst_features

In [ ]:
node_features = src_features.join(dst_features, how="outer")
node_features = node_features.fillna(0)

In [ ]:
for node, features in node_features.iterrows():
    if graph.has_node(node):
        graph.nodes[node].update(features.to_dict())

In [ ]:
print(graph.nodes["59.166.0.0"])

In [ ]:
node_features.head()

In [ ]:
# Add nodes with attributes
for node, features in node_features.iterrows():
    graph.add_node(node, **features.to_dict())

In [ ]:
print("Nodes:", graph.number_of_nodes())
print("Edges:", graph.number_of_edges())

print("Average degree:",
      sum(dict(graph.degree()).values()) / graph.number_of_nodes())

print("Self loops:",
      netx.number_of_selfloops(graph))

print("Connected Components:",
      netx.number_weakly_connected_components(graph))

In [ ]:
degrees = [d for _, d in graph.degree()]

import matplotlib.pyplot as plt

plt.hist(degrees, bins=30)
plt.xlabel("Node Degree")
plt.ylabel("Number of Nodes")
plt.title("Degree Distribution")
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as funct
from torch_geometric.nn import GCNConv

In [ ]:
print(data)

In [ ]:
from sklearn.preprocessing import StandardScaler

scalerNode = StandardScaler()

node_features[:] = scalerNode.fit_transform(node_features)

data.x = torch.tensor(
    node_features.values,
    dtype=torch.float
)

In [ ]:
class EdgeGCN(nn.Module):

    def __init__(
        self,
        node_feature_dim,
        edge_feature_dim,
        hidden_dim=64,
        embedding_dim=32
    ):
        super().__init__()

        # GCN
        self.conv1 = GCNConv(node_feature_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)

        # Edge Classifier
        self.classifier = nn.Sequential(

            nn.Linear(
                embedding_dim * 2 + edge_feature_dim,
                64
            ),

            nn.ReLU(),

            nn.Linear(64, 32),

            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self,x,edge_index,edge_attr):

        x = self.conv1(x, edge_index)
        x = funct.relu(x)

        x = self.conv2(x, edge_index)
        x = funct.relu(x)

        src = x[edge_index[0]]

        dst = x[edge_index[1]]

        edge_representation = torch.cat(

            [
                src,
                dst,
                edge_attr
            ],

            dim=1
        )

        out = self.classifier(edge_representation)

        return out

In [ ]:
model = EdgeGCN(

    node_feature_dim=data.x.shape[1],

    edge_feature_dim=data.edge_attr.shape[1]

).to(device)

In [ ]:
optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

criterion = nn.CrossEntropyLoss()

In [ ]:
data = data.to(device)

In [ ]:
epochs = 150
for epoch in range(epochs):

    model.train()

    optimizer.zero_grad(set_to_none=True)

    out = model(

        data.x,

        data.edge_index,

        data.edge_attr
    )

    loss = criterion(

        out[trainIndx],
        data.edge_label[trainIndx]
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1:03d}"
        f" Loss : {loss.item():.4f}"
    )

In [ ]:
model.eval()

with torch.no_grad():

    logits = model(

        data.x,

        data.edge_index,

        data.edge_attr
    )

    prediction = logits[testIndx].argmax(dim=1)


In [ ]:
correct = (

    prediction == data.edge_label[testIndx]

).sum()

accuracy = correct / len(data.edge_label[testIndx])

print(
    accuracy.item()
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print(classification_report(
    data.edge_label[testIndx].cpu(),
    prediction.cpu()
))

print(confusion_matrix(
    data.edge_label[testIndx].cpu(),
    prediction.cpu()
))

In [ ]:
out

In [ ]:
prob = torch.softmax(out[testIndx], dim=1)[:, 1]

print(
    roc_auc_score(
        data.edge_label[testIndx].detach().cpu().numpy(),
        prob.detach().cpu().numpy()
    )
)